In [164]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error

# PyTorch Forecasting + Lightning
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer, MultiNormalizer, TorchNormalizer
from pytorch_forecasting.metrics import SMAPE, RMSE, MAE, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.callbacks.progress import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("medium")   # utilise RTX Tensor Cores
    print("float32 matmul precision set to 'medium'")

# import pytorch_forecasting.models.base._base_model as _bm
# _orig_plot = _bm.BaseModel.log_prediction
#
# def _patched_log_prediction(self, x, out, batch_idx, **kwargs):
#     # cast any bf16 tensors in out to float32 before matplotlib touches them
#     out = {
#         k: v.float() if isinstance(v, torch.Tensor) and v.dtype == torch.bfloat16 else v
#         for k, v in out.items()
#     }
#     return _orig_plot(self, x, out, batch_idx, **kwargs)
#
# _bm.BaseModel.log_prediction = _patched_log_prediction

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision set to 'medium'


In [165]:
# ==============================================================================
# Cell 2 – Memory-Optimization Helpers  (unchanged from original script)
# ==============================================================================
def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        if np.iinfo(np.int8).min <= min_val <= max_val <= np.iinfo(np.int8).max:
            return "int8"
        if np.iinfo(np.int16).min <= min_val <= max_val <= np.iinfo(np.int16).max:
            return "int16"
        if np.iinfo(np.int32).min <= min_val <= max_val <= np.iinfo(np.int32).max:
            return "int32"
        return "int64"
    else:
        if 0 <= min_val <= max_val <= np.iinfo(np.uint8).max:
            return "uint8"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint16).max:
            return "uint16"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint32).max:
            return "uint32"
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Converts columns to smaller dtypes.
    For low-decimal float columns, stores scaled integers if that beats float32.
    Returns:
        optimized_df
        metadata dict with scaling info
    """
    meta = {}

    for col in df.columns:
        s = df[col]

        unique_non_null = set(df[col].dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}):
            if col == "Група":
                df[col] = s.astype("bool")
                meta[col] = {"stored_as": "bool", "scale": 1}
                continue

        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue

        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue

            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()

            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)
                mn = int(np.nanmin(scaled))
                mx = int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                int_bytes = np.dtype(int_dtype).itemsize
                float32_bytes = np.dtype("float32").itemsize
                if int_bytes < float32_bytes:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                else:
                    df[col] = s.astype("float32")
                    meta[col] = {"stored_as": "float32", "scale": 1}
            else:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}

    return df, meta

In [166]:
# ==============================================================================
# Cell 3 – Metric Helpers  (unchanged from original script)
# ==============================================================================
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [167]:
# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_PATH = "data/silver_money_calc/train.parquet"
VAL_PATH   = "data/silver_money_calc/val.parquet"
TEST_PATH  = "data/silver_money_calc/test.parquet"
TEST2_PATH  = "data/no_y_col/with_weather_v2.parquet"

# ── Target ─────────────────────────────────────────────────────────────────────
Y_COL = 'Sum of кВт'

GROUP_COL = "EIC-код_cat"

In [168]:
FUTURE_REALS = [
    'temperature_2m', 'apparent_temperature',
       'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid',
       'cloud_cover_high', 'surface_pressure', 'wind_speed_10m',
       'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation',
       'diffuse_radiation', 'direct_normal_irradiance'
]

# ── Numerical real-valued covariates NOT known at forecast time ─────────────────
PAST_REALS = ['Ціна розподілу ЕЕ', 'Ціна ЕЕ', "Money_spent"]   # add any columns that are only available historically

# ── Static numerical covariates (one value per station, time-invariant) ─────────
STATIC_REALS = [
    'Широта', 'Довгота',
]

# ── Categorical time-varying known covariates ───────────────────────────────────
# Calendar categoricals — always known in the future
TIME_VARYING_KNOWN_CATS = [
    # "Year_cat",
    "Month_cat", "Day_cat", "Hour_cat", "day_of_week_cat", "season_cat"
]  # created in Cell 5

# ── Static categoricals ─────────────────────────────────────────────────────────
# One-hot encoded columns will be collapsed back to a single categorical.
# Any remaining OHE groups (Тип, Область, …) are handled similarly.
STATIC_CATS = ['EIC-код_cat', 'Група_cat', 'АЗС_cat', 'Тип_cat', 'Область_cat', 'ОСР код_cat', 'ОСР опис_cat']

In [169]:
# ── Forecasting horizon & encoder length ───────────────────────────────────────
# Predict this many hours ahead
MAX_PREDICTION_LENGTH = 24          # 24 h forecast
MAX_ENCODER_LENGTH    = 36    # look-back window: 7 days

# ── Training hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE    = 1024 * 2
MAX_EPOCHS    = 50
LEARNING_RATE = 3e-3
HIDDEN_SIZE            = 32   # was 64
ATTENTION_HEAD_SIZE    = 2    # was 4
HIDDEN_CONTINUOUS_SIZE = 16   # was 32
DROPOUT       = 0.1

In [170]:
def build_time_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    PyTorch Forecasting requires a monotonically increasing integer time_idx
    that is *shared* across all groups (think of it as the global hour counter).
    """
    # Global hourly index starting from the minimum timestamp in the dataset

    df['datetime'] = pd.to_datetime(df['datetime'])
    min_dt = df["datetime"].min()
    try:
        if GLOBAL_MIN_DT:
            df["time_idx"] = ((df["datetime"] - GLOBAL_MIN_DT).dt.total_seconds() / 3600).astype(int)
    except:
        GLOBAL_MIN_DT = df["datetime"].min()
        df["time_idx"] = ((df["datetime"] - GLOBAL_MIN_DT).dt.total_seconds() / 3600).astype(int)
    return df

def add_cat_helpers(df: pd.DataFrame) -> pd.DataFrame:
    """Create string categoricals from datetime for PyTorch Forecasting."""

    # ensure datetime column is in datetime format
    df['datetime'] = pd.to_datetime(df['datetime'])

    # basic time components
    # df['Year_cat'] = df['datetime'].dt.year
    df['Month_cat'] = df['datetime'].dt.month.astype(str)
    df['Day_cat'] = df['datetime'].dt.day.astype(str)
    df['Hour_cat'] = df['datetime'].dt.hour.astype(str)

    # day of week (0=Monday)
    df['day_of_week_cat'] = df['datetime'].dt.dayofweek.astype(str)

    # season mapping
    def get_season(month):
        if month in [12, 1, 2]:
            return 'winter'
        elif month in [3, 4, 5]:
            return 'spring'
        elif month in [6, 7, 8]:
            return 'summer'
        else:
            return 'autumn'

    df['season_cat'] = df['datetime'].dt.month.map(get_season)

    return df

def load_and_prepare(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path).reset_index(drop=True)

    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)

    # Cast compressed ints back to float32 for PTF
    for col in df.select_dtypes(
        include=["int8","int16","int32","uint8","uint16","uint32"]
    ).columns:
        if col not in ["Year","Month","Day","Hour","day_of_week","season_number"]:
            df[col] = df[col].astype("float32")

    df = build_time_index(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith("_cat"):
            df[col] = df[col].astype(str)
    try:
        df[Y_COL] = df[Y_COL].astype("float32")
    except:
        df[Y_COL] = 0
        df[Y_COL] = df[Y_COL].astype("float32")
    df = df.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)

    return df

In [171]:
print("Loading train …")
train = load_and_prepare(TRAIN_PATH)

# Compute global min datetime from train so val/test share the same time_idx origin
# GLOBAL_MIN_DT = train["datetime"].min()

print("Loading val   …")
val = load_and_prepare(VAL_PATH)

print("Loading test  …")
test = load_and_prepare(TEST_PATH)

print("Loading test2  …")
test2 = load_and_prepare(TEST2_PATH)
test2['Ціна розподілу ЕЕ'] = 0
test2['Ціна ЕЕ'] = 0
test2['Money_spent'] = 0

Loading train …
Loading val   …
Loading test  …
Loading test2  …


In [172]:
station_stats = (
    train.groupby(GROUP_COL)
    .agg(
        rows=(Y_COL, "count"),
        hours_span=("time_idx", lambda x: x.max() - x.min()),
        missing_pct=("time_idx", lambda x: 1 - len(x) / (x.max() - x.min() + 1)),
        target_mean=(Y_COL, "mean"),
        target_std=(Y_COL, "std"),
        target_zeros=(Y_COL, lambda x: (x == 0).mean()),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

TARGET_STATIONS = 1

np.random.seed(42)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations kept: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test2 = test2[test2[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows     : {len(train):,}")
print(f"Train batches  : {len(train) // BATCH_SIZE}")

Stations kept: 1
Train rows     : 13,125
Train batches  : 6


In [173]:
# ==============================================================================
# Cell 6 – Build PyTorch Forecasting Datasets
# ==============================================================================

train["data_subset"] = "train"
val["data_subset"] = "val"
test["data_subset"] = "test"
test2["data_subset"] = "test2"

# Concatenate for creating the datasets (PTF handles the split internally)
train_val_data = (
    pd.concat([train, val, test, test2], ignore_index=True)
    .sort_values([GROUP_COL, "time_idx"])
    .reset_index(drop=True)
)

# --- force datetime column into one consistent pandas datetime dtype ---
train_val_data["datetime"] = pd.to_datetime(
    train_val_data["datetime"],
    utc=True,
    errors="coerce",
)

# optional sanity check
bad_dt = train_val_data["datetime"].isna().sum()
print(f"Bad datetime rows: {bad_dt}")

# make them timezone-naive after normalizing to UTC
train_val_data["datetime"] = train_val_data["datetime"].dt.tz_convert(None)

# rebuild global time_idx
train_val_data["time_idx"] = (
    (train_val_data["datetime"] - train_val_data["datetime"].min()).dt.total_seconds() // 3600
).astype("int64")

training_cutoff = train_val_data.loc[train_val_data["data_subset"] == "train", "time_idx"].max()
val_cutoff = train_val_data.loc[train_val_data["data_subset"] == "val", "time_idx"].max()
test_cutoff = train_val_data.loc[train_val_data["data_subset"] == "test", "time_idx"].max()

train_val_data.drop(columns=["data_subset"], inplace=True)

print(f"training cutoff: {training_cutoff}\nval cutoff: {val_cutoff}\ntest cutoff: {test_cutoff}")

# ── Sanity-check required columns exist ────────────────────────────────────────
missing = [
    c for c in FUTURE_REALS + PAST_REALS + STATIC_REALS
    + TIME_VARYING_KNOWN_CATS + STATIC_CATS
    if c not in train_val_data.columns
]
if missing:
    print("⚠️  Missing columns (remove from config or add to pre-processing):")
    for m in missing:
        print("   ", m)
else:
    print("✅  All configured columns present.")

def filter_existing(lst, df):
    return [c for c in lst if c in df.columns]

future_reals_ok = filter_existing(FUTURE_REALS, train_val_data)
past_reals_ok = filter_existing(PAST_REALS, train_val_data)
static_reals_ok = filter_existing(STATIC_REALS, train_val_data)
tv_known_cats_ok = filter_existing(TIME_VARYING_KNOWN_CATS, train_val_data)
static_cats_ok = filter_existing(STATIC_CATS, train_val_data)

training = TimeSeriesDataSet(
    train_val_data[train_val_data["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target=Y_COL,
    group_ids=[GROUP_COL],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=static_cats_ok,
    static_reals=static_reals_ok,
    time_varying_known_reals=future_reals_ok + ["time_idx"],
    time_varying_known_categoricals=tv_known_cats_ok,
    time_varying_unknown_reals=[Y_COL] + past_reals_ok,
    target_normalizer=GroupNormalizer(
        groups=[GROUP_COL], transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    train_val_data[train_val_data["time_idx"] <= val_cutoff],
    predict=True,
    stop_randomization=True,
)

Bad datetime rows: 0
training cutoff: 13125
val cutoff: 13869
test cutoff: 14614
✅  All configured columns present.


In [174]:
# ── DataLoaders ────────────────────────────────────────────────────────────────
NUM_WORKERS = 0

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    # remove persistent_workers and prefetch_factor when num_workers=0
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"\nTraining   batches : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")


Training   batches : 6
Validation batches : 1


In [175]:
# ==============================================================================
# Cell 7 – Build the TFT / LSTM Model
# ==============================================================================
# PyTorch Forecasting's TemporalFusionTransformer uses LSTMs internally as
# its sequence encoder/decoder – it is the recommended "LSTM +" architecture
# for multi-series panel forecasting with covariates.
# If you strictly need a vanilla LSTM, swap TFT for
# pytorch_forecasting.models.DeepAR or pytorch_forecasting.models.NHiTS.

from pytorch_forecasting.metrics import RMSE

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate          = LEARNING_RATE,
    hidden_size            = HIDDEN_SIZE,
    attention_head_size    = ATTENTION_HEAD_SIZE,
    dropout                = DROPOUT,
    hidden_continuous_size = HIDDEN_CONTINUOUS_SIZE,
    loss                   = RMSE(),        # ← was QuantileLoss()
    log_interval           = -1,
    optimizer              = "adam",
    reduce_on_plateau_patience = 4,
)

print(f"Number of parameters: {tft.size() / 1e3:.1f}k")

# tft = torch.compile(tft)

Number of parameters: 173.6k


In [176]:
early_stop = EarlyStopping(
    monitor   = "val_loss",
    min_delta = 1e-4,
    patience  = 5,
    verbose   = True,
    mode      = "min",
)

lr_logger = LearningRateMonitor()

checkpoint = ModelCheckpoint(
    dirpath   = "checkpoints/",
    filename  = "tft-{epoch:02d}-{val_loss:.4f}",
    monitor   = "val_loss",
    mode      = "min",
    save_top_k = 2,
)

# TQDMProgressBar renders correctly inside Jupyter notebooks.
# refresh_rate controls how often (in batches) the bar updates.
progress_bar = TQDMProgressBar(refresh_rate=20)

logger = TensorBoardLogger("tb_logs", name="tft_gas_stations")
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {accelerator} accelerator")

trainer = pl.Trainer(
    max_epochs           = MAX_EPOCHS,
    accelerator          = accelerator,
    devices              = 1,
    gradient_clip_val    = 0.1,
    callbacks            = [early_stop, lr_logger, checkpoint, progress_bar],
    logger               = logger,
    enable_progress_bar  = True,
    log_every_n_steps    = 10,
    precision="bf16-mixed",
    val_check_interval=0.25,           # validate 4× per epoch instead of once
    accumulate_grad_batches=4,
)

Using bfloat16 Automatic Mixed Precision (AMP)


Using gpu accelerator


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [177]:
# ==============================================================================
# Cell 9 – Train
# ==============================================================================

trainer.fit(
    tft,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)

print(f"\nBest checkpoint : {checkpoint.best_model_path}")
print(f"Best val_loss   : {checkpoint.best_model_score:.6f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ RMSE                            │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    683 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    928 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 11.4 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 60.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 48.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  3.2 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 173 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 173 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 952                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.750. Signaling Trainer to stop.



Best checkpoint : C:\Users\Lev\Documents\GitHub\Diploma\checkpoints\tft-epoch=00-val_loss=0.7500.ckpt
Best val_loss   : 0.750015


In [178]:
# ==============================================================================
# Cell 10 – Load Best Checkpoint & Validate
# ==============================================================================

best_tft = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint.best_model_path,
    weights_only=False,
)
best_tft.eval()

result = best_tft.predict(
    val_loader,
    return_index=True,
    trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"},
)

# result[0]: (100, 24) — one row per station, 24 prediction steps
# result[2]: (100, 2)  — one row per station with GROUP_COL + time_idx (start of window)
pred_tensor = result[0].cpu().numpy()  # (100, 24)
val_index = result[2]  # (100, 2)

# Expand index: for each station repeat across all 24 prediction steps
rows = []
for i, (_, idx_row) in enumerate(val_index.iterrows()):
    for step in range(MAX_PREDICTION_LENGTH):
        rows.append({
            GROUP_COL: idx_row[GROUP_COL],
            "time_idx": idx_row["time_idx"] + step,
            "pred": pred_tensor[i, step],
        })

val_expanded = pd.DataFrame(rows)

# Merge with ground truth
val_merged = val_expanded.merge(
    train_val_data[[GROUP_COL, "time_idx", Y_COL]],
    on=[GROUP_COL, "time_idx"],
    how="inner",
)

val_pred = val_merged["pred"].values
val_true = val_merged[Y_COL].values

print(f"Aligned samples: {len(val_true)}")
print("\n── Validation Metrics ──────────────────────────────────────")
print(f"SMAPE : {smape(val_true, val_pred):.4f}")
print(f"RMSE  : {rmse(val_true, val_pred):.4f}")
print(f"MAPE  : {mape(val_true, val_pred):.2f} %")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Aligned samples: 26

── Validation Metrics ──────────────────────────────────────
SMAPE : 0.2200
RMSE  : 3.6420
MAPE  : 30.77 %


In [184]:
full_dataset = TimeSeriesDataSet.from_dataset(
    training,
    train_val_data,
    predict=True,           # only the prediction window beyond training_cutoff
    stop_randomization=True,
)

full_loader = full_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

full_result = best_tft.predict(
    full_loader,
    return_index=True,
    trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"},
)

# result[0]: (100, 24) — one row per station, 24 prediction steps
# result[2]: (100, 2)  — one row per station with GROUP_COL + time_idx (start of window)
full_pred_tensor = full_result[0].cpu().numpy()  # (100, 24)
full_index = full_result[2]  # (100, 2)

# Expand index: for each station repeat across all 24 prediction steps
rows = []
for i, (_, idx_row) in enumerate(full_index.iterrows()):
    for step in range(MAX_PREDICTION_LENGTH):
        rows.append({
            GROUP_COL: idx_row[GROUP_COL],
            "time_idx": idx_row["time_idx"] + step,
            "pred": full_pred_tensor[i, step],
        })

full_expanded = pd.DataFrame(rows)

# Merge with ground truth
full_merged = full_expanded.merge(
    train_val_data[[GROUP_COL, "time_idx", Y_COL, 'datetime']],
    on=[GROUP_COL, "time_idx"],
    how="inner",
)

val_merged = full_merged[(full_merged['time_idx'] <= val_cutoff) & (full_merged['time_idx'] > training_cutoff)].copy()
test_merged = full_merged[(full_merged['time_idx'] <= test_cutoff) & (full_merged['time_idx'] > val_cutoff)].copy()
test2_merged = full_merged[full_merged['time_idx'] > test_cutoff].copy()

val_pred = val_merged["pred"].values
val_true = val_merged[Y_COL].values

print(f"Aligned samples: {len(val_true)}")
print("\n── Validation Metrics ──────────────────────────────────────")
print(f"SMAPE : {smape(val_true, val_pred):.4f}")
print(f"RMSE  : {rmse(val_true, val_pred):.4f}")
print(f"MAPE  : {mape(val_true, val_pred):.2f} %")

test_pred = test_merged["pred"].values
test_true = test_merged[Y_COL].values

print(f"Aligned samples: {len(test_true)}")
print("\n── Validation Metrics ──────────────────────────────────────")
print(f"SMAPE : {smape(test_true, test_pred):.4f}")
print(f"RMSE  : {rmse(test_true, test_pred):.4f}")
print(f"MAPE  : {mape(test_true, test_pred):.2f} %")


test2_pred = test2_merged["pred"].values
test2_true = test2_merged[Y_COL].values

print(f"Aligned samples: {len(test2_true)}")
print("\n── Validation Metrics ──────────────────────────────────────")
print(f"SMAPE : {smape(test2_true, test2_pred):.4f}")
print(f"RMSE  : {rmse(test2_true, test2_pred):.4f}")
print(f"MAPE  : {mape(test2_true, test2_pred):.2f} %")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Aligned samples: 0

── Validation Metrics ──────────────────────────────────────
SMAPE : nan


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [185]:
training_cutoff

13125

In [186]:
val_cutoff

13869

In [187]:
test_cutoff

14614

In [188]:
full_merged

,EIC-код_cat,time_idx,pred,Sum of кВт,datetime
0,62Z7973386804189,19679,12.417747,0.0,2026-03-30 22:00:00
1,62Z7973386804189,19680,12.410135,0.0,2026-03-30 23:00:00
2,62Z7973386804189,19681,12.914180,0.0,2026-03-31 00:00:00
3,62Z7973386804189,19682,12.702251,0.0,2026-03-31 01:00:00
4,62Z7973386804189,19683,13.076984,0.0,2026-03-31 02:00:00
5,62Z7973386804189,19684,13.080558,0.0,2026-03-31 03:00:00
6,62Z7973386804189,19685,12.861390,0.0,2026-03-31 04:00:00
7,62Z7973386804189,19686,12.527991,0.0,2026-03-31 05:00:00
8,62Z7973386804189,19687,13.478008,0.0,2026-03-31 06:00:00
9,62Z7973386804189,19688,13.525664,0.0,2026-03-31 07:00:00


In [111]:




test2_dataset = TimeSeriesDataSet.from_dataset(
    training,
    test2,
    predict=True,           # only the prediction window beyond training_cutoff
    stop_randomization=True,
)

test2_loader = test2_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test2_result = best_tft.predict(
    test2_loader,
    return_index=True,
    trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"},
)

test2_pred_tensor = test2_result[0].cpu().numpy()  # (100, 24)
test2_index = test2_result[2]  # (100, 2)

rows = []
for i, (_, idx_row) in enumerate(test2_index.iterrows()):
    for step in range(MAX_PREDICTION_LENGTH):
        rows.append({
            GROUP_COL: idx_row[GROUP_COL],
            "time_idx": idx_row["time_idx"] + step,
            "pred": test2_pred_tensor[i, step],
        })

test2_expanded = pd.DataFrame(rows)

# Merge with ground truth
test2_merged = test2_expanded.merge(
    test2[[GROUP_COL, "time_idx", Y_COL]],
    on=[GROUP_COL, "time_idx"],
    how="inner",
)

test2_pred = test2_merged["pred"].values
test2_true = test2_merged[Y_COL].values

print(f"Aligned samples: {len(test2_true)}")
print("\n── Validation Metrics ──────────────────────────────────────")
print(f"SMAPE : {smape(test2_true, test2_pred):.4f}")
print(f"RMSE  : {rmse(test2_true, test2_pred):.4f}")
print(f"MAPE  : {mape(test2_true, test2_pred):.2f} %")

Loading test2  …


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


NaN 0 rows
Aligned samples: 24

── Validation Metrics ──────────────────────────────────────
SMAPE : 2.0000
RMSE  : 10.2701
MAPE  : nan %
